# 🚀 실전 Seq2Seq 번역기 아키텍처 및 LSTM 병목 분석

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## 1. nn.LSTM을 활용한 Seq2Seq 모델 구조화
현업에서는 밑바닥부터 짜지 않고 최적화된 C++ 백엔드의 `nn.LSTM`을 사용합니다. 인코더의 마지막 상태가 디코더의 초기 상태로 넘어가는 '컨텍스트 벡터' 흐름을 확인합니다.

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: 모든 시점의 상태
        # hidden, cell: 문장을 다 읽고 난 후의 최종 상태 (이것이 컨텍스트 벡터!)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, input, hidden, cell):
        # input: 단일 단어 (batch_size, 1)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

print("✅ Seq2Seq 아키텍처(Encoder-Decoder) 모듈화 완료")

✅ Seq2Seq 아키텍처(Encoder-Decoder) 모듈화 완료


## 2. 병목(Bottleneck) 현상 실험
문장의 길이가 10단어일 때와 1000단어일 때, 인코더가 뱉어내는 컨텍스트 벡터의 크기(메모리 용량)가 늘어나는지 실험해 봅니다.

In [3]:
vocab_size, emb_dim, hidden_dim = 5000, 256, 512
encoder = Encoder(vocab_size, emb_dim, hidden_dim)

# 시나리오 A: 10개의 단어로 이루어진 짧은 문장
short_sentence = torch.randint(0, vocab_size, (32, 10)) # batch=32, seq_len=10
_, (hidden_short, cell_short) = encoder(short_sentence)

# 시나리오 B: 1000개의 단어로 이루어진 거대한 논문 텍스트
long_sentence = torch.randint(0, vocab_size, (32, 1000)) # batch=32, seq_len=1000
_, (hidden_long, cell_long) = encoder(long_sentence)

print(f"짧은 문장의 컨텍스트 벡터 크기: {hidden_short.shape}")
print(f"긴 문장의 컨텍스트 벡터 크기:   {hidden_long.shape}")

if hidden_short.shape == hidden_long.shape:
    print("\n🚨 [치명적 문제 발견] 문장 길이가 100배 차이 나는데도, 정보를 담아두는 그릇(Hidden Vector)의 크기는 동일합니다!")
    print("이것이 바로 Seq2Seq의 정보 병목 현상입니다. 앞부분에 읽었던 정보는 필연적으로 찌그러지거나 소실됩니다.")
    print("이 한계를 부수기 위해 '어텐션(Attention)' 메커니즘이 탄생했습니다.")

ValueError: not enough values to unpack (expected 2, got 1)